# Document Graph Demo

This notebook shows how to use `genai-graph` to build a **generic Document+Chunk knowledge graph** from a directory of text files.

## What you'll do
1. Configure a directory and graph output path
2. Run the `DocumentDirectoryFactory` to ingest files → Document nodes
3. (Optional) Add Chunk nodes via semantic chunking
4. Query the resulting graph with Cypher
5. Visualize the graph inline

**No LLM needed** for the basic graph — just files and the graph engine.

> To extend with entity extraction, see `DocumentDirectoryFactory` docstring for subclassing instructions.

## 1. Configuration

Edit the paths below to point at your documents directory and desired output location.

In [ ]:
import tempfile
from pathlib import Path

# ── Edit these paths ──────────────────────────────────────────────────────
# Directory containing documents to ingest (markdown / text files)
DOCUMENTS_DIR = Path("docs")  # genai-graph docs as a sample corpus

# Where to store the Kuzu graph database
DB_PATH = Path(tempfile.mkdtemp()) / "document_graph.db"
# ─────────────────────────────────────────────────────────────────────────

DOCUMENTS_DIR = DOCUMENTS_DIR.resolve()
print(f"Documents : {DOCUMENTS_DIR}")
print(f"Graph DB  : {DB_PATH}")
print(f"Files found: {len(list(DOCUMENTS_DIR.rglob('*.md')))} markdown files")

: 

## 2. Build the Document Graph

The `DocumentDirectoryFactory` scans the directory and creates:
- One **Document** node per file (path, filename, size, hash, mime-type)
- **Chunk** nodes (semantic chunks using *chonkie*) linked by `CONTAINS` and `NEXT`

In [ ]:
from genai_graph.kg.backend import KuzuBackend
from genai_graph.kg.factories.document_factory import DocumentDirectoryFactory
from genai_graph.kg.ingest.extract import create_schema
from genai_graph.kg.ingest.merge import merge_nodes_batch, merge_relationships_batch

# 1. Open (or create) the graph database
backend = KuzuBackend()
backend.connect(str(DB_PATH))
print(f"✅ Opened graph DB at {DB_PATH}")

# 2. Initialise the factory
factory = DocumentDirectoryFactory(
    data_root=str(DOCUMENTS_DIR),
    include=["*.md", "*.txt", "*.rst"],
    chunk_size=512,
    overlap=50,
)

schema = factory.build_schema()
print(f"Schema nodes     : {[n.node_class.__name__ for n in schema.nodes]}")
print(f"Schema relations : {[r.name for r in schema.relations]}")

In [ ]:
# 3. Create table schema in Kuzu
create_schema(backend, schema.nodes, schema.relations)
print("✅ Schema created")

In [ ]:
from genai_graph.kg.nodes.document import Chunk, Document

all_doc_nodes = []
all_chunk_nodes = []
all_contains_rels = []  # Document → Chunk
all_next_rels = []  # Chunk → Chunk

keys = factory.get_keys()
print(f"Ingesting {len(keys)} file(s) …")

for file_path in keys:
    doc = factory.get_struct_data_by_key(file_path)
    if doc is None:
        continue
    all_doc_nodes.append(doc)

    chunks = factory.build_document_chunks(file_path)
    for i, chunk in enumerate(chunks):
        all_chunk_nodes.append(chunk)
        all_contains_rels.append((doc.path, chunk.chunk_id))  # Document → Chunk
        if i > 0:
            all_next_rels.append((chunks[i - 1].chunk_id, chunk.chunk_id))  # Chunk → Chunk

print(f"  Documents : {len(all_doc_nodes)}")
print(f"  Chunks    : {len(all_chunk_nodes)}")
print(f"  CONTAINS  : {len(all_contains_rels)}")
print(f"  NEXT      : {len(all_next_rels)}")

In [ ]:
from datetime import datetime

from genai_graph.kg.ingest.extract import RelationshipRecord, import_neo4j_data
from genai_graph.kg.ingest.merge import NodeDataCollection

now = datetime.utcnow().isoformat() + "Z"

nodes_data = NodeDataCollection()
for doc in all_doc_nodes:
    d = doc.model_dump()
    d["name"] = doc.filename
    d["_created_at"] = now
    d["_updated_at"] = now
    nodes_data.add("Document", d)

for chunk in all_chunk_nodes:
    c = chunk.model_dump()
    c.pop("embedding", None)  # skip None embeddings (avoid FLOAT[] type conflict)
    c["name"] = chunk.chunk_id
    c["_created_at"] = now
    c["_updated_at"] = now
    nodes_data.add("Chunk", c)

relationships = []
for doc_path, chunk_id in all_contains_rels:
    relationships.append(RelationshipRecord("Document", doc_path, "Chunk", chunk_id, "CONTAINS", {}))
for from_cid, to_cid in all_next_rels:
    relationships.append(RelationshipRecord("Chunk", from_cid, "Chunk", to_cid, "NEXT", {}))

import_neo4j_data(backend, nodes_data, relationships, key_fields={"Document": "path", "Chunk": "chunk_id"})
print(f"✅ Graph written: {nodes_data.total_count()} nodes, {len(relationships)} relationships")

## 3. Query the Graph

Use standard Cypher to explore the graph.

In [ ]:
from rich.console import Console
from rich.table import Table

console = Console()


def run_query(cypher: str, title: str = "Results") -> None:
    """Execute a Cypher query and display as a Rich table."""
    try:
        df = backend.execute_get_as_df(cypher, union=True)
        if df.empty:
            console.print(f"[yellow]{title}: no results[/yellow]")
            return
        table = Table(title=f"{title} ({len(df)} rows)")
        for col in df.columns:
            table.add_column(str(col), style="cyan")
        for _, row in df.head(20).iterrows():
            table.add_row(*[str(v) for v in row])
        console.print(table)
        if len(df) > 20:
            console.print(f"[dim]… {len(df) - 20} more rows[/dim]")
    except Exception as exc:
        console.print(f"[red]Query error: {exc}[/red]")

In [ ]:
# Node counts by type
run_query("MATCH (d:Document) RETURN d.filename, d.file_size, d.mime_type ORDER BY d.filename", "Documents")

In [ ]:
# Chunks per document
run_query(
    """
    MATCH (d:Document)-[:CONTAINS]->(c:Chunk)
    RETURN d.filename AS document, count(c) AS chunks
    ORDER BY chunks DESC
    """,
    "Chunks per Document",
)

In [ ]:
# Show first chunk of each document
run_query(
    """
    MATCH (d:Document)-[:CONTAINS]->(c:Chunk)
    WHERE c.chunk_index = 0
    RETURN d.filename AS document, c.text AS first_chunk
    ORDER BY d.filename
    """,
    "First Chunk per Document",
)

## 4. Visualize the Graph

Generate an interactive HTML graph and display it inline.

In [ ]:
import sys
import tempfile
from pathlib import Path

from genai_graph.kg.export.html import generate_html

# Fetch a compact subgraph for visualization (documents + first chunk of each)
cypher = """
MATCH (d:Document)-[:CONTAINS]->(c:Chunk)
WHERE c.chunk_index = 0
RETURN d, c
LIMIT 50
"""

try:
    html_content = generate_html(
        connection=backend,
        destination_file_path="/tmp/document_graph.html",
        query=cypher,
    )
    _html_path = Path(tempfile.mkdtemp()) / "kg_data.html"
    _html_path.write_text(html_content, encoding="utf-8")
    if "ipykernel" in sys.modules:
        from IPython.display import HTML, display  # noqa: PLC0415

        display(HTML(html_content))
    else:
        print(f"Graph HTML saved: {_html_path}")
except Exception as exc:
    print(f"Graph HTML not available: {exc}")
    print("Use 'cli kg view' after running the document_graph workflow for a full visualization.")

## 5. Schema Visualization

Display the schema (node types and relationships) as an interactive D3 diagram.

In [ ]:
import sys
import tempfile
from pathlib import Path

schema = factory.build_schema()

try:
    from genai_graph.kg.schema import ResolvedSchema

    resolved = ResolvedSchema.from_graph_schema(schema)
    schema_html = resolved.to_html()
    _schema_path = Path(tempfile.mkdtemp()) / "schema.html"
    _schema_path.write_text(schema_html, encoding="utf-8")
    if "ipykernel" in sys.modules:
        from IPython.display import HTML, display  # noqa: PLC0415

        display(HTML(schema_html))
    else:
        print(f"Schema HTML saved: {_schema_path}")
    print("\nNode types:")
    for node in schema.nodes:
        print(f"  - {node.label}")
    print("\nRelationships:")
    for rel in schema.relations:
        print(f"  - {rel.from_node.label} -[{rel.name}]-> {rel.to_node.label}")
except Exception as exc:
    print(f"Schema visualization not available: {exc}")

## Alternative: Use the CLI Workflow

Instead of the manual steps above, you can use the bundled `document_graph` workflow:

```bash
# Dry-run to see the plan
uv run cli workflow run document_graph --dry-run

# Run with default settings (scans ${paths.data_root}/documents)
uv run cli workflow run document_graph

# Run with custom directory
uv run cli workflow run document_graph --set data_dir=/path/to/docs

# Open the HTML visualization
uv run cli kg view
```

The workflow handles schema creation, ingestion, and HTML export automatically.

## Next Steps: Entity Extraction

To add LLM-based entity extraction on top of the Document+Chunk graph:

1. Define a BAML schema with your domain entities
2. Subclass `DocumentDirectoryFactory` and override `build_schema()` to add your entity nodes
3. Override `get_struct_data_by_key()` to call BAML extraction per document

See `genai_graph/kg/factories/document_factory.py` docstring and the `ekg-atos` project
for a full example of this pattern.